# PyTorch DTensor 学习笔记

DTensor（Distributed Tensor）用统一的张量布局抽象，描述数据如何在多个进程与设备之间分片、复制或等待归约。它是构建张量并行、DDP、FSDP 与组合并行的重要基础。

## 运行方式

DTensor 示例需要多个进程协同执行，建议从终端使用 `torchrun` 启动，而不是逐个运行 notebook 单元：

```powershell
torchrun --standalone --nnodes=1 --nproc_per_node=4 dtensor_example.py --launcher torchrun
```

- `--standalone`：单机启动，自动设置 rendezvous。
- `--nnodes=1`：只有一台机器。
- `--nproc_per_node=4`：每台机器启动 4 个进程，通常对应 4 张 GPU。

## 1. 为什么需要 DTensor？

大模型训练通常组合三类并行策略：

| 策略 | 主要切分对象 | 常见 PyTorch 方案 |
| --- | --- | --- |
| 数据并行（Data Parallel） | 数据批次 | DDP、FSDP |
| 张量并行（Tensor Parallel） | 层内张量与参数 | DTensor、Tensor Parallel API |
| 流水线并行（Pipeline Parallel） | 模型层 | PiPPy 等 |

DTensor 提供共同的 `DeviceMesh + placements` 抽象，使上述策略能够组合，并清楚描述一个全局张量在各设备上的物理存储布局。

### DTensor 的价值

1. 统一复杂并行布局下的 `state_dict` 保存与加载。
2. 在 eager mode（即时执行模式）中支持灵活的张量并行。
3. 作为 SPMD（Single Program, Multiple Data，单程序多数据）编程模型和编译器分布式训练的基础构件。

## 2. 核心概念：DeviceMesh 与 Placement

`DeviceMesh` 定义参与计算的进程/GPU 拓扑；`placements` 则逐个对应 mesh 维度，指定张量沿该维度如何分布。

| Placement | 含义 |
| --- | --- |
| `Shard(dim)` | 将张量的第 `dim` 维切分到不同设备。 |
| `Replicate()` | 每个设备保存完整副本。 |
| `Partial(reduce_op)` | 每个设备保存局部计算结果，尚未跨设备归约。 |

### Placement 限制：不能混用 Partial 归约类型

同一 DTensor 中的所有 `Partial` 必须使用相同归约操作。`sum` 与 `max` 的执行顺序不可交换，若混用会造成语义歧义，因此 DTensor 会抛出 `ValueError`。

In [ ]:
from torch.distributed.tensor import Partial, Shard

# 合法：所有 Partial 都执行 sum 归约。
valid_placements = [Partial("sum"), Partial("sum"), Shard(0)]

# 不合法：sum 与 max 的归约顺序会影响结果。
invalid_placements = [Partial("sum"), Partial("max"), Shard(0)]

## 3. 基础 DTensor API

基础 API 用于：

- 从全局 `torch.Tensor` 创建 DTensor：`distribute_tensor`。
- 从各 rank 已持有的本地张量创建 DTensor：`DTensor.from_local`。
- 在保持逻辑全局张量不变的前提下调整布局：`redistribute`。

In [ ]:
import torch
from torch.distributed.tensor import DTensor, Replicate, Shard, distribute_tensor, init_device_mesh

# 4 个 rank 组成一维 mesh：[0, 1, 2, 3]。
mesh_1d = init_device_mesh("cuda", (4,))
big_tensor = torch.randn(8, 4, device="cuda")

# 沿全局张量第 0 维切分：每个 rank 保存部分行。
row_sharded = distribute_tensor(big_tensor, mesh_1d, [Shard(0)])

# 沿全局张量第 1 维切分：每个 rank 保存部分列。
column_sharded = distribute_tensor(big_tensor, mesh_1d, [Shard(1)])

# 完整复制：每个 rank 保存完整张量。
replicated = distribute_tensor(big_tensor, mesh_1d, [Replicate()])

# 重新分布：行分片 -> 列分片。
column_sharded = row_sharded.redistribute(mesh_1d, [Shard(1)])

# from_local：假设当前 rank 已经持有它应保存的局部行分片。
local_tensor = torch.randn(2, 4, device="cuda", requires_grad=True)
from_local = DTensor.from_local(local_tensor, mesh_1d, [Shard(0)])

### 二维 mesh：复制与分片组合

对 `mesh = (2, 2)`，逻辑拓扑为：

```text
[[rank 0, rank 1],
 [rank 2, rank 3]]
```

`[Replicate(), Shard(0)]` 表示：先沿 mesh 第 0 维复制，再沿 mesh 第 1 维按张量第 0 维切分。于是 rank 0 与 2 保存相同的前半部分，rank 1 与 3 保存相同的后半部分。

In [ ]:
mesh_2d = init_device_mesh("cuda", (2, 2))
replicate_then_row_shard = [Replicate(), Shard(0)]
two_dim_tensor = distribute_tensor(big_tensor, mesh_2d, replicate_then_row_shard)

## 4. 模块级 API：distribute_module

对于已有参数的模块（如 `nn.Linear`），`distribute_module` 可借助三个回调完成模块级分布：

| 回调 | 作用 |
| --- | --- |
| `partition_fn` | 将模块参数/缓冲区原地替换为 DTensor。 |
| `input_fn` | 在前向前，将普通输入转换为所需 DTensor 布局；返回新的输入元组。 |
| `output_fn` | 在前向后处理 DTensor 输出；可返回完整普通 Tensor。 |

在官方当前实现中，`input_fn` 和 `output_fn` 的返回值会分别作为 `forward_pre_hook` 的新输入、`forward_hook` 的新输出。

In [ ]:
import torch.nn as nn
from torch.distributed.tensor import distribute_module

class MyModule(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(8, 8)

    def forward(self, x):
        return self.fc(x)

def shard_params(name, module, mesh):
    # 将 Linear 的输出特征维切分：每个 rank 保存部分 weight/bias。
    if isinstance(module, nn.Linear):
        for parameter_name, parameter in module.named_parameters(recurse=False):
            module.register_parameter(
                parameter_name,
                nn.Parameter(distribute_tensor(parameter, mesh, [Shard(0)])),
            )

def to_replicated_input(module, inputs, mesh):
    # 返回的新元组会替换真正传入 forward 的 inputs。
    return tuple(
        distribute_tensor(x, mesh, [Replicate()]) if isinstance(x, torch.Tensor) else x
        for x in inputs
    )

def to_full_output(module, output, mesh):
    # all_gather 并返回完整普通 Tensor。
    return output.full_tensor()

sharded_module = distribute_module(
    MyModule().cuda(),
    mesh_1d,
    partition_fn=shard_params,
    input_fn=to_replicated_input,
    output_fn=to_full_output,
)

# 调用方传入普通 Tensor；最终得到的也是完整普通 Tensor。
output = sharded_module(torch.randn(2, 8, device="cuda"))

## 5. 2.13 API 文档补充

本节整理 [torch.distributed.tensor 2.13 API 文档](https://docs.pytorch.ac.cn/docs/2.13/distributed.tensor.html#devicemesh-as-the-distributed-communicator) 的核心内容。该页面强调：`DeviceMesh` 是 DTensor 的分布式通信器；DTensor 的 `placements` 则描述张量在这个通信器上的布局。

### DeviceMesh 作为分布式通信器

创建 mesh 是 SPMD 操作：所有参与 rank 必须以相同的 `mesh_shape`、`mesh_dim_names` 和 rank 排列创建它，否则可能发生静默错误或通信挂起。`init_device_mesh()` 是推荐入口；当默认进程组尚未初始化且环境变量已准备好时，它可以自动建立默认进程组。

| API | 用途 |
| --- | --- |
| `init_device_mesh(device_type, mesh_shape, mesh_dim_names=...)` | 创建 mesh。 |
| `mesh.get_group(mesh_dim)` | 获取当前 rank 在某个 mesh 维度所属的 `ProcessGroup`。 |
| `mesh.get_coordinate()` | 获取当前 rank 在多维 mesh 中的坐标。 |
| `mesh.get_local_rank(mesh_dim)` | 获取当前 rank 在指定 mesh 维度中的局部 rank。 |
| `mesh.size(mesh_dim=None)` | 获取整体或指定维度的大小。 |
| `mesh["name"]` / `mesh["a", "b"]` | 以已命名维度切出一维/多维子 mesh。 |

In [ ]:
from torch.distributed.device_mesh import init_device_mesh

# 使用 4 个进程时，所有 rank 都必须执行完全相同的建 mesh 语句。
mesh = init_device_mesh(
    "cuda", (2, 2), mesh_dim_names=("replicate", "shard")
)

coordinate = mesh.get_coordinate()
replicate_group = mesh.get_group("replicate")
shard_group = mesh.get_group("shard")
shard_mesh = mesh["shard"]

### DTensor 生命周期 API

| API | 输入数据的含义 | 输出 / 通信 |
| --- | --- | --- |
| `dtensor.__create_chunk_list__()` | DTensor。 | 返回当前 rank 本地块的 `ChunkStorageMetadata`（大小、全局偏移）；主要供 Distributed Checkpoint 使用。 |
| `distribute_tensor(tensor, mesh, placements)` | 通常 rank 0 持有完整全局张量。 | 按 placements 分发为 DTensor。 |
| `DTensor.from_local(local_tensor, mesh, placements)` | 每个 rank 已持有自己的物理局部张量。 | 将局部张量包装为 DTensor。 |
| `dtensor.to_local()` | DTensor。 | 返回当前 rank 的局部普通 Tensor；保留 autograd 连接。 |
| `dtensor.full_tensor()` | DTensor。 | 聚合为每个 rank 都拥有的完整普通 Tensor。 |
| `dtensor.redistribute(mesh, placements)` | DTensor 与目标布局。 | 返回新布局 DTensor；必要时触发 all-gather、reduce-scatter 或 all-reduce。 |

`__create_chunk_list__()` 名称以双下划线包围，但它是公开文档列出的检查点接口，而非 Python 魔术方法；一个 DTensor 在每个 rank 通常只返回一个本地块。

`Partial("sum")` 表示一个尚未归约的局部结果。将其重新分布为 `Replicate()` 时会执行 `sum` 归约。所有 `Partial` placement 必须使用相同归约类型。

### 分布式工厂函数与随机数

`torch.distributed.tensor` 提供 `ones`、`zeros`、`empty`、`full`、`rand`、`randn`，可直接创建 DTensor，无需先构造完整普通 Tensor。

随机工厂函数具有分布式 RNG 语义：分片布局会得到互不重复的随机值，复制布局会得到相同随机值。调用前所有参与 rank 必须具有一致 RNG 状态，调用后状态也会同步推进；随机 DTensor 当前不支持 CPU 后端。

In [ ]:
from torch.distributed.tensor import empty, full, ones, rand, randn, zeros

torch.manual_seed(2026)  # 所有 rank 必须以相同种子进入随机 DTensor 操作。
placements = [Shard(0)]
x1 = ones(8, 4, device_mesh=mesh_1d, placements=placements)
x2 = zeros(8, 4, device_mesh=mesh_1d, placements=placements)
x3 = empty(8, 4, device_mesh=mesh_1d, placements=placements)
x4 = full(8, 4, fill_value=3.14, device_mesh=mesh_1d, placements=placements)
x5 = rand(8, 4, device_mesh=mesh_1d, placements=placements)
x6 = randn(8, 4, device_mesh=mesh_1d, placements=placements)

### 页面 API 完整清单（不筛选）

以下为该页面列出的全部名称；对应代码示例均放在同目录 `dtensor_example.py`，其中以下划线开头的 Placement 类和 `experimental` 命名空间属于内部/实验接口。

```text
DTensor：DTensor、__create_chunk_list__、device_mesh、placements、from_local、to_local、full_tensor、redistribute
创建/模块：distribute_tensor、distribute_module、ones、zeros、empty、full、rand、randn
Placement：Placement、is_shard、is_replicate、is_partial、Shard、local_shard_size_and_offset、Replicate、Partial、ALL_REDUCE_OPS、LINEAR_REDUCE_OPS
内部 Placement：_StridedShard、split_factor、local_shard_size_and_offset、_MaskPartial、mask_buffer、offset_shape、offset_dim
调试：CommDebugMode、get_comm_counts、get_total_counts、get_parameter_info、get_sharding_info、generate_comm_debug_tracing_table、generate_json_dump、log_comm_debug_tracing_table_to_file、visualize_sharding
实验：context_parallel、local_map、register_sharding、implicit_replication
```

注意：`_StridedShard`、`_MaskPartial` 以 `_` 开头，主要服务 DTensor/FSDP 内部布局；实验 API 可能在后续版本修改。

### 混合 Tensor/DTensor、调试与实验 API

DTensor 运算不允许混合普通 `torch.Tensor` 与 DTensor，因为普通 Tensor 在各 rank 是否相同的语义不明确。若普通张量确实在所有 rank 相同，应显式包装为 `Replicate()` DTensor。

```python
same_on_all_ranks = DTensor.from_local(torch.arange(8, device="cuda"), mesh_1d, [Replicate()])
result = same_on_all_ranks + dtensor
```

调试时可设置 `TORCH_LOGS=+dtensor` 查看详细日志；`CommDebugMode` 可统计上下文内的集合通信，`visualize_sharding()` 可在终端显示一维或二维 DTensor 分片。文档还列出实验性 `local_map`、`context_parallel`、`register_sharding` 与 `implicit_replication`；这些 API 可能发生变化，应以当前安装版本的文档为准。

## 下一步

请运行同目录的 `dtensor_example.py` 查看每个 rank 的实际局部张量、参数分片和 `all_gather` 后的完整输出。建议依次观察：`Shard(0)`、`Shard(1)`、`Replicate()`、二维 mesh，以及 `distribute_module` 的输入/输出 hook。